In [17]:
# definimos las librerias
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

In [18]:
# creamos la sesion
spark = (SparkSession.builder
         .appName("aplicacion")
         .master("local[*]")
         .config("spark.driver.memory", "2g")
         .getOrCreate()
)
print(f"versión: {spark.version}")

versión: 4.2.0


In [19]:
# creamos schema
schema = StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("country", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("category", StringType(), True),
    StructField("date", StringType(), True),
    StructField("_reg_corrupt", StringType(), True)
])

In [20]:
# lectura + definimos método de valores corruptos: permission
df_csv = spark.read.csv("p2_transactions.csv", header=True, schema=schema, sep=",",
                        mode="PERMISSIVE", columnNameOfCorruptRecord="_reg_corrupt")
df_csv.show(5)

+--------------+-----------+-------+------+-----------+----------+------------+
|transaction_id|customer_id|country|amount|   category|      date|_reg_corrupt|
+--------------+-----------+-------+------+-----------+----------+------------+
|             1|        101|  Spain| 120.5|Electronics|2024-01-15|        NULL|
|             2|        102| France|  75.0|    Fashion|2024-01-17|        NULL|
|             3|        103|Germany| 210.3|       Home|2024-01-20|        NULL|
|             4|        104|  Spain|  33.0|      Books|2024-02-02|        NULL|
|             5|        105| France| 450.0|Electronics|2024-02-05|        NULL|
+--------------+-----------+-------+------+-----------+----------+------------+
only showing top 5 rows


In [21]:
# transformamos fecha a tipo -> timestampt
df_csv = (df_csv
          .withColumn("date_ts", F.to_timestamp(F.col("date")))
          .withColumn("year", F.year(F.col("date_ts")))
          .withColumn("month", F.month(F.col("date_ts")))
)

df_csv.show(5)

+--------------+-----------+-------+------+-----------+----------+------------+-------------------+----+-----+
|transaction_id|customer_id|country|amount|   category|      date|_reg_corrupt|            date_ts|year|month|
+--------------+-----------+-------+------+-----------+----------+------------+-------------------+----+-----+
|             1|        101|  Spain| 120.5|Electronics|2024-01-15|        NULL|2024-01-15 00:00:00|2024|    1|
|             2|        102| France|  75.0|    Fashion|2024-01-17|        NULL|2024-01-17 00:00:00|2024|    1|
|             3|        103|Germany| 210.3|       Home|2024-01-20|        NULL|2024-01-20 00:00:00|2024|    1|
|             4|        104|  Spain|  33.0|      Books|2024-02-02|        NULL|2024-02-02 00:00:00|2024|    2|
|             5|        105| France| 450.0|Electronics|2024-02-05|        NULL|2024-02-05 00:00:00|2024|    2|
+--------------+-----------+-------+------+-----------+----------+------------+-------------------+----+-----+
o

In [22]:
# definimos el particionador para parquet
(df_csv.write.mode("overwrite") # si existe algo, lo sobreescribimos
 .partitionBy("year","month") # manera en la que particionamos el dataset
 .option("compression", "snappy") # manera de comprimir los ficheros
 .parquet("03_directorios_parquet") # ruta donde se almacenan
)

# lectura del dataset en parquet
df_parquet = spark.read.parquet("03_directorios_parquet")
df_parquet.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- date: string (nullable = true)
 |-- _reg_corrupt: string (nullable = true)
 |-- date_ts: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



In [23]:
# filtramos por mes y año
df_filtrado = df_parquet.filter((F.col("year") == 2024) & (F.col("month") == 2))
df_filtrado.count()

17

In [24]:
df_filtrado.show(5)

+--------------+-----------+-------+------+-----------+----------+------------+-------------------+----+-----+
|transaction_id|customer_id|country|amount|   category|      date|_reg_corrupt|            date_ts|year|month|
+--------------+-----------+-------+------+-----------+----------+------------+-------------------+----+-----+
|             4|        104|  Spain|  33.0|      Books|2024-02-02|        NULL|2024-02-02 00:00:00|2024|    2|
|             5|        105| France| 450.0|Electronics|2024-02-05|        NULL|2024-02-05 00:00:00|2024|    2|
|             6|        106|Germany|  89.9|    Fashion|2024-02-06|        NULL|2024-02-06 00:00:00|2024|    2|
|             7|        107|  Italy|  25.0|      Books|2024-02-07|        NULL|2024-02-07 00:00:00|2024|    2|
|             8|        108|  Spain| 310.0|       Home|2024-02-08|        NULL|2024-02-08 00:00:00|2024|    2|
+--------------+-----------+-------+------+-----------+----------+------------+-------------------+----+-----+
o

In [ ]:
spark.stop()